# Text Splitter Comparison

## Purpose

This notebook compares different LangChain text splitting strategies for the markdown document corpus. The current system uses `RecursiveCharacterTextSplitter` with generic separators. Since all sagemaker-docs are markdown files, markdown-aware splitters may produce higher-quality chunks that better preserve document structure.

## Splitters Compared

| Splitter | Approach | Key Difference |
|----------|----------|---------------|
| **RecursiveCharacterTextSplitter** (baseline) | Generic separators: `\n\n`, `\n`, ` `, `""` | Current production splitter |
| **RecursiveCharacterTextSplitter (Markdown)** | Markdown-aware separators: headers, code fences, lists | Respects markdown structure in split decisions |
| **MarkdownHeaderTextSplitter** | Splits on `#`/`##`/`###` headers, preserves hierarchy | Structurally-driven chunks with header context |

## Two-Part Analysis

| Part | What It Tests | API Required? |
|------|--------------|---------------|
| **Part A: Local Chunk Analysis** | Chunk statistics, size distributions, example chunks | No |
| **Part B: Search Quality** | Index pre-chunked content, run queries, compare results | Yes |

## Prerequisites

- FastAPI server running on `localhost:8000` (for Part B only)
- OpenSearch index populated with sagemaker-docs
- `pip install -r requirements.txt`

## Important Notes

- **Part B runtime**: ~15-30 minutes total (each splitter requires a full clear-index-search cycle)
- **Part B is destructive**: It clears and re-indexes the OpenSearch index multiple times
- **Restoration**: The final cell restores the default configuration

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import requests

# Add experiments/ to path so helpers can be imported
sys.path.insert(0, ".")
from helpers import (
    search,
    results_to_dataframe,
    hits_to_doc_ids,
    jaccard_similarity,
    overlap_matrix,
    rank_biased_overlap,
    clear_index,
    reindex_local_docs,
    get_index_stats,
    run_query_suite,
    BASE_URL,
)

# LangChain splitters
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    Language,
)

# Verify the API is reachable
stats = get_index_stats()
print(f"Index: {stats['index_name']}")
print(f"Documents: {stats['doc_count']}")
print(f"Status: {stats['status']}")

In [ ]:
# Load all sagemaker-docs content locally
DOCS_DIR = Path("../sagemaker-docs")
docs = {}
for p in sorted(DOCS_DIR.glob("*.md")):
    docs[p.name] = p.read_text(encoding="utf-8", errors="replace")

total_chars = sum(len(v) for v in docs.values())
print(f"Loaded {len(docs)} documents")
print(f"Total characters: {total_chars:,}")
print(f"Average doc length: {total_chars // len(docs):,} chars")

## Splitter Definitions

All splitters use `chunk_size=500` and `chunk_overlap=50` (matching current production defaults) for a fair comparison. The `MarkdownHeaderTextSplitter` splits on headers first, then applies a secondary `RecursiveCharacterTextSplitter` to any sections that exceed the chunk size.

In [ ]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50


def split_recursive_char(text: str) -> list[str]:
    """Baseline: RecursiveCharacterTextSplitter with default separators."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    return splitter.split_text(text)


def split_markdown_recursive(text: str) -> list[str]:
    """Markdown-aware recursive splitter using Language.MARKDOWN separators."""
    splitter = RecursiveCharacterTextSplitter.from_language(
        language=Language.MARKDOWN,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    return splitter.split_text(text)


def split_markdown_header(text: str) -> list[str]:
    """Split on markdown headers, preserving hierarchy as context prefix."""
    headers_to_split_on = [
        ("#", "H1"),
        ("##", "H2"),
        ("###", "H3"),
    ]
    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on,
    )
    secondary_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )

    header_docs = header_splitter.split_text(text)
    chunks = []
    for doc in header_docs:
        # Prepend header hierarchy as context
        header_parts = []
        for key in sorted(doc.metadata.keys()):
            if doc.metadata.get(key):
                header_parts.append(doc.metadata[key])
        header_prefix = " > ".join(header_parts)
        content = f"{header_prefix}\n\n{doc.page_content}" if header_prefix else doc.page_content

        # Apply secondary split if content exceeds chunk_size
        if len(content) > CHUNK_SIZE:
            chunks.extend(secondary_splitter.split_text(content))
        else:
            chunks.append(content)
    return chunks


SPLITTERS = {
    "recursive_char": split_recursive_char,
    "markdown_recursive": split_markdown_recursive,
    "markdown_header": split_markdown_header,
}

print(f"Splitters defined: {list(SPLITTERS.keys())}")
print(f"Chunk size: {CHUNK_SIZE}, Chunk overlap: {CHUNK_OVERLAP}")

---
## Part A: Local Chunk Analysis

Apply all splitters to all documents locally and compare chunk statistics. No API calls needed for this section.

In [ ]:
# Apply all splitters to all documents
all_chunks = {}  # (splitter_name, filename) -> list[str]

for splitter_name, splitter_fn in SPLITTERS.items():
    print(f"Chunking with {splitter_name}...")
    for filename, content in docs.items():
        chunks = splitter_fn(content)
        all_chunks[(splitter_name, filename)] = chunks
    total = sum(len(all_chunks[(splitter_name, f)]) for f in docs)
    print(f"  Total chunks: {total}")

print("\nDone.")

### Chunk Statistics Summary

Compare total chunks, average chunk size, and size variability across splitters.

In [ ]:
summary_data = []
for splitter_name in SPLITTERS:
    all_lens = []
    doc_chunk_counts = []
    for filename in docs:
        chunks = all_chunks[(splitter_name, filename)]
        doc_chunk_counts.append(len(chunks))
        all_lens.extend(len(c) for c in chunks)
    summary_data.append({
        "Splitter": splitter_name,
        "Total Chunks": sum(doc_chunk_counts),
        "Avg Chunks/Doc": round(np.mean(doc_chunk_counts), 1),
        "Avg Chunk Len": int(round(np.mean(all_lens))),
        "Median Chunk Len": int(round(np.median(all_lens))),
        "Min Chunk Len": min(all_lens),
        "Max Chunk Len": max(all_lens),
        "Std Chunk Len": int(round(np.std(all_lens))),
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

### Chunk Size Distributions

Histograms showing the distribution of chunk lengths for each splitter. A tight distribution means consistent chunk sizes; a wide distribution may indicate the splitter produces a mix of small and large chunks.

In [ ]:
fig, axes = plt.subplots(1, len(SPLITTERS), figsize=(5 * len(SPLITTERS), 4), sharey=True)
colors = ["#4C72B0", "#55A868", "#C44E52"]

for ax, (splitter_name, color) in zip(axes, zip(SPLITTERS.keys(), colors)):
    all_lens = []
    for filename in docs:
        all_lens.extend(len(c) for c in all_chunks[(splitter_name, filename)])

    ax.hist(all_lens, bins=50, color=color, alpha=0.7, edgecolor="black", linewidth=0.5)
    ax.set_title(splitter_name)
    ax.set_xlabel("Chunk Length (chars)")
    ax.axvline(CHUNK_SIZE, color="red", linestyle="--", linewidth=1, label=f"target={CHUNK_SIZE}")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Count")
plt.suptitle("Chunk Size Distributions by Splitter", fontsize=14)
plt.tight_layout()
plt.show()

### Example Chunks

Compare the first 3 chunks from the same document under each splitter. This helps visualize how each splitter decides where to cut.

In [ ]:
# Pick a representative document (one with headers and some structure)
EXAMPLE_DOC = "sagemaker-roles.md"
NUM_EXAMPLE_CHUNKS = 3

print(f"Example document: {EXAMPLE_DOC}")
print(f"Document length: {len(docs[EXAMPLE_DOC]):,} chars")
print()

for splitter_name in SPLITTERS:
    chunks = all_chunks[(splitter_name, EXAMPLE_DOC)]
    print(f"{'='*80}")
    print(f"Splitter: {splitter_name} ({len(chunks)} total chunks)")
    print(f"{'='*80}")
    for i, chunk in enumerate(chunks[:NUM_EXAMPLE_CHUNKS]):
        print(f"\n--- Chunk {i + 1} ({len(chunk)} chars) ---")
        # Show first 300 chars of each chunk
        preview = chunk[:300] + "..." if len(chunk) > 300 else chunk
        print(preview)
    print()

### Part A Observations

_Fill in after running the notebook._

1. **Which splitter produces the most/fewest chunks?**
   - _TODO_

2. **How do chunk size distributions compare?**
   - _TODO: Does the MarkdownHeaderTextSplitter produce more variable chunk sizes?_

3. **Qualitative chunk quality:**
   - _TODO: Do the markdown-aware splitters produce more coherent chunks?_
   - _TODO: Does header context preservation help or add noise?_

---
## Part B: Search Quality Comparison

For each splitter, we index the pre-chunked content into OpenSearch and compare search results.

### Approach

Since the API always uses `RecursiveCharacterTextSplitter` internally, we use a workaround:

1. Chunk all documents locally with splitter X
2. Index each chunk as a separate "document" via `POST /opensearch/bulk-index`
3. Set `chunk_size=99999` so the API's internal splitter doesn't re-split our chunks
4. Use synthetic filenames (`original.md::splitter_name::0001`) to avoid dedup collisions

The OpenSearch ingest pipeline auto-embeds each chunk's content, so search works normally.

In [ ]:
# Test queries — semantic and mixed queries that are most sensitive to chunking quality
TEST_QUERIES = [
    "How do I make sure my notebook isn't exposed to the internet?",
    "What is the benefit of using a project instead of running pipelines directly?",
    "How can data scientists share code consistently across a team?",
    "What IAM permissions does an execution role need to run a training job?",
    "How does EventBridge trigger actions when an endpoint changes status?",
    "What kubectl commands do I use to check a training job running in Kubernetes?",
]


def parse_synthetic_filename(synthetic: str) -> tuple[str, str, int]:
    """Parse 'original.md::splitter::0003' -> (original.md, splitter, 3)."""
    parts = synthetic.rsplit("::", 2)
    if len(parts) == 3:
        return parts[0], parts[1], int(parts[2])
    return synthetic, "unknown", 0


print(f"Test queries: {len(TEST_QUERIES)}")

In [ ]:
# Index pre-chunked documents for each splitter and run queries
partb_results = {}  # splitter_name -> {query -> response}
partb_stats = {}    # splitter_name -> {total_chunks, ...}
partb_timings = {}  # splitter_name -> seconds

for splitter_name in SPLITTERS:
    print(f"\n{'='*60}")
    print(f"Splitter: {splitter_name}")
    print(f"{'='*60}")

    start = time.time()

    # Step 1: Clear the index
    print("  Clearing index...")
    clear_index()

    # Step 2: Build bulk document list from pre-chunked content
    print("  Building pre-chunked document list...")
    bulk_docs = []
    for filename in docs:
        chunks = all_chunks[(splitter_name, filename)]
        for i, chunk_text in enumerate(chunks):
            bulk_docs.append({
                "filename": f"{filename}::{splitter_name}::{i:04d}",
                "content": chunk_text,
            })
    print(f"  Total chunks to index: {len(bulk_docs)}")

    # Step 3: Index via bulk-index with chunk_size=99999 to prevent re-splitting
    print("  Indexing (this may take several minutes)...")
    resp = requests.post(
        f"{BASE_URL}/opensearch/bulk-index",
        json={
            "documents": bulk_docs,
            "chunk_size": 99999,
            "chunk_overlap": 0,
            "max_concurrency": 10,
        },
        timeout=1200,
    )
    resp.raise_for_status()
    index_result = resp.json()

    # Step 4: Wait for OpenSearch to settle
    time.sleep(5)

    stats = get_index_stats()
    partb_stats[splitter_name] = {
        "total_chunks": stats["doc_count"],
        "indexed_count": index_result["indexed_count"],
    }
    print(f"  Indexed: {stats['doc_count']} total chunks")

    # Step 5: Run queries
    print("  Running queries...")
    partb_results[splitter_name] = run_query_suite(
        TEST_QUERIES, search_type="hybrid", size=10,
    )

    elapsed = time.time() - start
    partb_timings[splitter_name] = elapsed
    print(f"  Done in {elapsed:.1f}s")

print(f"\nTotal Part B time: {sum(partb_timings.values()):.1f}s")

### Part B: Side-by-Side Results

Compare the top-5 results for each query across splitter configurations. The `filename` column shows the original source document (parsed from the synthetic filename).

In [ ]:
splitter_names = list(SPLITTERS.keys())

for query in TEST_QUERIES:
    print(f"\n{'='*90}")
    print(f"Query: {query}")
    print(f"{'='*90}")
    for splitter_name in splitter_names:
        resp = partb_results[splitter_name][query]
        df = results_to_dataframe(resp)
        # Parse synthetic filenames to show original document name
        df["source_file"] = df["filename"].apply(
            lambda f: parse_synthetic_filename(f)[0]
        )
        print(f"\n--- {splitter_name} ({resp['total_hits']} total hits) ---")
        display(df[["rank", "score", "source_file", "content_preview"]].head(5))

### Part B: Result Set Overlap Between Splitters

In [ ]:
# Since doc_ids will be completely different between splitter configs
# (different chunks = different OpenSearch doc IDs), we compare using
# the source filename instead of doc_id.

jaccard_matrices = []
rbo_data = []

for query in TEST_QUERIES:
    # Extract source filenames (not synthetic) for set-based comparison
    source_files = {}
    for splitter_name in splitter_names:
        resp = partb_results[splitter_name][query]
        files = [
            parse_synthetic_filename(hit["filename"])[0]
            for hit in resp.get("hits", [])
        ]
        source_files[splitter_name] = files

    jm = overlap_matrix(source_files)
    jaccard_matrices.append(jm)

    # Pairwise RBO on source filenames
    for i, a in enumerate(splitter_names):
        for b in splitter_names[i + 1:]:
            rbo_val = rank_biased_overlap(source_files[a], source_files[b])
            rbo_data.append({
                "query": query[:50],
                "pair": f"{a} vs {b}",
                "rbo": round(rbo_val, 3),
            })

avg_jaccard = sum(jaccard_matrices) / len(jaccard_matrices)
print("Average Jaccard Similarity Matrix (source filenames, across all queries):")
display(avg_jaccard.round(3))

In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(avg_jaccard.values, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(len(splitter_names)))
ax.set_yticks(range(len(splitter_names)))
ax.set_xticklabels(splitter_names, rotation=30, ha="right")
ax.set_yticklabels(splitter_names)
for i in range(len(splitter_names)):
    for j in range(len(splitter_names)):
        ax.text(j, i, f"{avg_jaccard.values[i, j]:.3f}", ha="center", va="center", fontsize=12)
plt.colorbar(im, ax=ax, label="Jaccard Similarity")
ax.set_title("Result Set Overlap Between Splitters (by Source File)")
plt.tight_layout()
plt.show()

# RBO table
rbo_df = pd.DataFrame(rbo_data)
print("\nRank-Biased Overlap (RBO, p=0.9) per query:")
display(rbo_df.pivot(index="query", columns="pair", values="rbo").round(3))

### Part B: Score Distribution

In [ ]:
score_data = []
for splitter_name in splitter_names:
    for query in TEST_QUERIES:
        resp = partb_results[splitter_name][query]
        for hit in resp.get("hits", []):
            score_data.append({
                "splitter": splitter_name,
                "score": hit["score"],
            })

score_df = pd.DataFrame(score_data)

fig, ax = plt.subplots(figsize=(10, 5))
score_df.boxplot(column="score", by="splitter", ax=ax)
ax.set_title("Score Distributions by Splitter")
ax.set_xlabel("Splitter")
ax.set_ylabel("Score")
plt.suptitle("")
plt.tight_layout()
plt.show()

---
## Restore Default Index

Re-index with the current production defaults so the system is left in a usable state.

In [ ]:
print("Restoring default index configuration (chunk_size=500, chunk_overlap=50)...")
clear_index()
reindex_local_docs(chunk_size=500, chunk_overlap=50)
stats = get_index_stats()
print(f"Restored: {stats['doc_count']} chunks indexed")

---
## Observations and Takeaways

_Fill in after running the notebook with actual results._

### Part A: Chunk Quality

1. **Do markdown-aware splitters produce meaningfully different chunks?**
   - _TODO_

2. **How do chunk size distributions differ?**
   - _TODO: Does MarkdownHeaderTextSplitter produce more variable chunk sizes?_

3. **Does header context preservation help?**
   - _TODO: Do the header-prefixed chunks from MarkdownHeaderTextSplitter contain useful context?_

### Part B: Search Quality

4. **Which splitter produced the most relevant search results?**
   - _TODO_

5. **How much result overlap exists between splitters?**
   - _TODO: High overlap means the splitter choice doesn't matter much_

6. **Is there a clear winner, or are the differences marginal?**
   - _TODO_

### Recommendation

7. **Should the project switch from the generic RecursiveCharacterTextSplitter to a markdown-aware alternative?**
   - _TODO: Weigh search quality improvement against the complexity of changing the pipeline_